### Structured Output

Models can be requested to provide their response in format matching a given schema.This is useful for ensuring the output can be easily parsed and used in subsequent processing.Langchain supports multiple schema types and methods for ensuring structured output.

### Pydantic 

Pydantic models provide the richest feature set with field validation, description and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model

model = init_chat_model(model="mistral-medium-3-5")
response = model.invoke("what is 2+2?")

response

AIMessage(content='The sum of 2 + 2 is **4**.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 22, 'total_tokens': 35, 'completion_tokens': 13, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-3-5', 'model': 'mistral-medium-3-5', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019effbd-e2ec-7aa0-b71d-586925329e30-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 13, 'total_tokens': 35})

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="the title of the movie")
    languages: str = Field(description="langueages in which movied was released originally")
    year: int = Field(description="the year in which movie was released")
    caste: str = Field(description="the actors, actoress in the movie")
    director: str = Field(description="name of the director of the movie")
    ratings: int = Field(description="IMDB ratings for the movie")

In [7]:
model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide me details about the movie Interstellar")

In [8]:
response

Movie(title='Interstellar', languages='English', year=2014, caste='Matthew McConaughey, Anne Hathaway, Jessica Chastain, Michael Caine, Matt Damon, Casey Affleck, Tars (voice by Bill Irwin)', director='Christopher Nolan', ratings=8)

In [19]:
class WorldCupMatch(BaseModel):
    winner: str = Field(description="name of the team who won the match")
    rankings: list[int] = Field(description="global FIFA rankings of the teams")
    teams:list [str] = Field(description="name of the teams who played")
    score: str = Field(description="goals scored by both the teams")
    goalScorers: str = Field(description="name of the players who scored goals accompanied by the timestamps at which they scored")
    date: str = Field(description="date when the event happened")
    stadium: str = Field(description="stadium's location at which the match happened")

football_model = model.with_structured_output(WorldCupMatch)
response = football_model.invoke("Fifa 2022 Wc final?")
response
    

WorldCupMatch(winner='Argentina', rankings=[3, 4], teams=['Argentina', 'France'], score='3-3 (4-2 on penalties)', goalScorers="Lionel Messi (23', 108'), Ángel Di María (36'), Kylian Mbappé (80', 81', 118')", date='2022-12-18', stadium='Lusail Stadium, Lusail')

In [20]:
model.profile

### TypeDict ###

Typedict provides a simple alternative using Python built in typing, ideal when you dont need runtime validation.

In [26]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """a movie with details"""
    title: Annotated [str,..., "the title of the movie"]
    year: Annotated [int, ..., "the year the movie was released"]
    cast: Annotated [list [str], ...,"the cast of the movie with the name of role they played"]
    director: Annotated [str, ..., "the director of the movie"]
    rating: Annotated [float, ..., "The movies IMDB ratings"]

movie_dict= model.with_structured_output(MovieDict)
response = movie_dict.invoke("Give me details about Avengers 2")
response

{'title': 'Avengers: Age of Ultron',
 'year': 2015,
 'cast': ['Robert Downey Jr. as Tony Stark / Iron Man',
  'Chris Evans as Steve Rogers / Captain America',
  'Mark Ruffalo as Bruce Banner / Hulk',
  'Chris Hemsworth as Thor',
  'Scarlett Johansson as Natasha Romanoff / Black Widow',
  'Jeremy Renner as Clint Barton / Hawkeye',
  'James Spader as Ultron',
  'Samuel L. Jackson as Nick Fury',
  'Don Cheadle as James Rhodes / War Machine',
  'Aaron Taylor-Johnson as Pietro Maximoff / Quicksilver',
  'Elizabeth Olsen as Wanda Maximoff / Scarlet Witch',
  'Paul Bettany as J.A.R.V.I.S. / Vision'],
 'director': 'Joss Whedon',
 'rating': 7.3}

In [27]:
model.profile

### DataClasses

A data class is a class typically mainly data, although there arent really any restrictions. You create it using @dataclass decorator

In [ ]:
# using Pydnatic


from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="the name of the person")
    email: str = Field(description = "the email address of the person")
    phone: str = Field(description = "the phone number of the person")

agent = create_agent(
    model = "mistral-medium-3-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages" : [{"role":"user", "content":"Extract contact info from: John Doe, john@mail.com, (555) 123-4567"}]
})

print(result)
print(result["structured_response"])

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@mail.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='df757611-97d6-4eaa-ae1d-7728a226f806'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dH7yFhwto', 'type': 'function', 'function': {'name': 'ContactInfo', 'arguments': '{"name": "John Doe", "email": "john@mail.com", "phone": "(555) 123-4567"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 157, 'total_tokens': 197, 'completion_tokens': 40, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-3-5', 'model': 'mistral-medium-3-5', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019f0010-ee7b-7f62-9cb4-b712bdea3e34-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@mail.com', 'phone': '(555) 123-4567'}, 'id': 'dH7yFhwto', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens

In [34]:
# Using TypedDict

from typing_extensions import TypedDict, Annotated

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str = Field(description="the name of the person")
    email: str = Field(description = "the email address of the person")
    phone: str = Field(description = "the phone number of the person")

agent = create_agent(
    model = "mistral-medium-3-5",
    response_format=ContactInfo
)
result = agent.invoke({
    "messages" : [{"role":"user", "content":"Extract contact info from: John Doe, john@mail.com, (555) 123-4567"}]
})

print(result)
print(result["structured_response"])

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@mail.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='95bc4ad8-2ae1-4901-a8aa-003cccde93e4'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qn5A7hflC', 'type': 'function', 'function': {'name': 'ContactInfo', 'arguments': '{"name": "John Doe", "email": "john@mail.com", "phone": "(555) 123-4567"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 125, 'total_tokens': 165, 'completion_tokens': 40, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-3-5', 'model': 'mistral-medium-3-5', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019f002a-6df6-7583-8a92-028ce5b92717-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@mail.com', 'phone': '(555) 123-4567'}, 'id': 'qn5A7hflC', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_tokens

In [38]:
# using DataClass

from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information of a person"""
    name: str # the name of the person
    email: str # email of the person
    phone: str # phone number of the person

agent = create_agent(
    model = "mistral-medium-3-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact information from: Sara Lahm, sara@mail.com,(555) 123-4567"}]
})
result["structured_response"]

ContactInfo(name='Sara Lahm', email='sara@mail.com', phone='(555) 123-4567')